# ThinCurr Python Example: DIII-D VDE Reconstruction Create JAMfit Model

In this example we demonstrate how to use JAMfit.py for creating and preparing for a filament reconstruction for a more complex case - in this case we will be reconstructing a DIII-D VDE event using one of the solvers bundled within JAMfit. 



## Importing Libraries, ThinCurr, and Tokamaker

To load the JAMfit ThinCurr module, you will need to define where ThinCurr is located. This can be done through the PYTHONPATH enviornment variable or within the script using sys.path.append() as shown below, where we look for the environment variable OFT_ROOTPATH to point the path to where the OpenFUSIONToolkit is installed. If this enviornment variable is not defined, you can do so within the script as well by setting:
 
 
 ```os.environ["OFT_ROOTPATH"] = \path\to\OpenFUSIONToolkit\install_release```


In [ ]:
import os 
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path as pathlib_create 
import xml.etree.ElementTree as ET
from IPython.display import clear_output

## DELETE THIS BEFORE LAUNCHING ## 
home_dir = os.path.expanduser("~")
oft_root_path = os.path.join(home_dir, "OpenFUSIONToolkit/install_release")
os.environ["OFT_ROOTPATH"] = oft_root_path
## END DELETE BLOCK ## 

thincurr_python_path = os.getenv('OFT_ROOTPATH')
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit._core import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.JAMfit import JAMfit
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors, circular_flux_loop

from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh

myOFT = OFT_env(nthreads=4)

## User Inputs 

Here we define the relevant input files necessary for JAMfit, namely the thincurr mesfile (`thincurr_meshfile`), the xml file with the shaping and solenoid coil definitions (`xml_coils`), and a save path directory (`save_path_dir`) to save relevant plotting outputs. 

In [ ]:
meshfile_thincurr = "DIIID_meshfile_thincurr.h5"
meshfile_tokamaker = "DIIID_tokamaker_JAMfit_model.h5" 
xml_coils = "DIIID_JAMfit_ex.xml"
xml_name = "DIIID_JAMfit_wfil_ex.xml" 
save_path_dir = "DIIID_ex_plot_files"
folder_path = pathlib_create(save_path_dir)
folder_path.mkdir(parents=True, exist_ok=True)

## Defining Plasma Filament Points 

### Helper Functions for Placing Filaments
These helper functions are for placing filaments given a defined limiter region, which we will load using TokaMaker. 

In [ ]:
def point_to_segment_dist(px, py, x0, y0, x1, y1):
    """Distance from point (px,py) to segment (x0,y0)-(x1,y1)."""
    dx, dy = x1 - x0, y1 - y0
    if dx == dy == 0:
        # segment is a point
        return np.hypot(px - x0, py - y0)
    t = ((px - x0) * dx + (py - y0) * dy) / (dx*dx + dy*dy)
    t = np.clip(t, 0, 1)
    proj_x = x0 + t*dx
    proj_y = y0 + t*dy
    return np.hypot(px - proj_x, py - proj_y)

def construct_filaments_equal_spacing(limiter, n=10, tol=1e-1, z_buffer=1.5):
    """
    Places points within an enclosed limiter, ensures equal R and Z spacing.
    
    Input:
        limiter : array
            Array of (R, Z) points defining the limiter contour.
    """
    R_boundary, Z_boundary = limiter[:,0], limiter[:,1]

    
    # Bounding box
    Rmin, Rmax = np.min(R_boundary), np.max(R_boundary)
    Zmin, Zmax = np.min(Z_boundary), np.max(Z_boundary)

    # Apply Z buffer (safe for negative Zmin)
    Zmin_clipped = Zmin + z_buffer
    Zmax_clipped = Zmax - z_buffer
    z_mask = (Z_boundary >= Zmin_clipped) & (Z_boundary <= Zmax_clipped)
    Rmin = np.min(R_boundary[z_mask]) if np.any(z_mask) else np.min(R_boundary)
    Rmax = np.max(R_boundary[z_mask]) if np.any(z_mask) else np.max(R_boundary)
    
    z_core_mask = (Z_boundary >= Zmin_clipped) & (Z_boundary <= Zmax_clipped)
    Rmin = np.min(R_boundary[z_core_mask])
    if Zmin_clipped >= Zmax_clipped:
        raise ValueError(f"z_buffer={z_buffer} is too large — clipped range is empty "
                         f"(Zmin={Zmin:.3f}, Zmax={Zmax:.3f})")

    # Derive spacing from R, apply to clipped Z range
    dR = (Rmax - Rmin) / (n - 1)
    nZ = max(2, round((Zmax_clipped - Zmin_clipped) / dR) + 1)

    # Build grid over clipped Z range directly
    from matplotlib.path import Path
    Rg = np.linspace(Rmin, Rmax, n)
    Zg = np.linspace(Zmin_clipped, Zmax_clipped, nZ)
    RR, ZZ = np.meshgrid(Rg, Zg)
    points = np.column_stack([RR.ravel(), ZZ.ravel()])

    # Close polygon if needed
    R_bound = R_boundary.copy()
    Z_bound = Z_boundary.copy()
    if R_bound[0] != R_bound[-1] or Z_bound[0] != Z_bound[-1]:
        R_bound = np.append(R_bound, R_bound[0])
        Z_bound = np.append(Z_bound, Z_bound[0])

    polygon = Path(np.column_stack([R_bound, Z_bound]))
    mask = polygon.contains_points(points)

    inside_points = points[mask]
    #inside_points = points 
    if tol > 0.0:
        keep = []
        for r, z in inside_points:
            min_dist = np.min([
                point_to_segment_dist(r, z, R_bound[i], Z_bound[i],
                                      R_bound[i+1], Z_bound[i+1])
                for i in range(len(R_bound) - 1)
            ])
            if min_dist >= tol:
                keep.append([r, z])
        inside_points = np.array(keep)
    print(f"Rmin is: {Rmin}")
    print(f"np.min(R_boundary) is: {np.min(R_boundary)}")
    return inside_points[:,0], inside_points[:,1]

def get_grid_from_xml(xml_path): ## for this to work the filaments cannot be named! 
    tree = ET.parse(xml_path)
    root = tree.getroot()

    r_grid = []
    z_grid = [] 
    for coil_set in root.iter('coil_set'):
        if coil_set.get('name') is None:  # unnamed only
            for coil in coil_set.findall('coil'):
                r, z = [float(v.strip()) for v in coil.text.split(',')]
                r_grid.append(r)
                z_grid.append(z)

    return r_grid, z_grid 

## Placing Filaments Within The DIII-D Limiter 

In this cell we intialize the TokaMaker mesh in order to grab the limiter points, such that we can place plasma filaments within the limiter using the helper functions that we defined above. 

In [ ]:
mygs = TokaMaker(myOFT)
mesh_pts,mesh_lc,mesh_reg,coil_dict,cond_dict = load_gs_mesh(meshfile_tokamaker)
mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
mygs.setup_regions(cond_dict=cond_dict,coil_dict=coil_dict)
mygs.setup(order = 2, F0=1*1) # here we put an arbitrary multiplication for F0 as we only need the limiter 
limiter = mygs.lim_contour

Here we define our resolution to be 10, with a small tolerance of 1e-1 from the wall. With our helper function, this places 101 filaments with equal spacing within the limiter. 

In [ ]:
resolution = 10
tol = 1e-1
zbuffer = 0
  
R, Z = construct_filaments_equal_spacing(limiter, n=resolution, tol=tol, z_buffer=zbuffer)
points = np.column_stack([R, Z])

edit_xml = ET.parse(xml_coils)
root = edit_xml.getroot()
icoil_tag = root.find('./thincurr/icoils')
for r, z in points:
    coil_element = ET.SubElement(icoil_tag, 'coil_set')
    coil = ET.SubElement(coil_element, "coil")
    coil.set("scale", "1.0") #default scale value is 1.0
    coil.text = '{0:.6f}, {1:.6f}'.format(r, z)
ET.indent(edit_xml, space="  ", level=0)
edit_xml.write(xml_name, encoding="utf-8", xml_declaration=True)   
print("XML file updated with filament points.") 

fig, ax = plt.subplots(1,1) 
plt.plot(limiter[:,0],limiter[:,1],color='k') 
plt.scatter(R, Z, cmap='plasma', marker='o', edgecolors='k')
plt.xlabel('r')
plt.ylabel('z')
plt.gca().set_aspect('equal', adjustable='box')
plt.show()


print(f"number of points: {len(points)}")

## Creating a JAMfit Object

Now that we have finished creating a new XML, which contains both the shaping coils (in DIII-D's case, the F and E coils) and the plasma filament points, we can now use that xml to create a new JAMfit object. We also need to input our DIII-D ThinCurr mesh, as well as an ```oft_env``` variable, which we defined earlier. 

>**Note:** Once you define a JAMfit object, you will still need to run \ref OpenFUSIONToolkit.ThinCurr.JAMfit.setup_JAMfit "setup_JAMfit()" which intializes the magnetic sensors. In this case, we have already provided them in the "floops.loc" file, and thus simply need to load them in. Look in \ref OpenFUSIONToolkit.ThinCurr.sensor for more specific details on how to build magnetic sensors or in other examples. 


>**Note:** This "floops.loc" also places a nonphysical flux loop at every point within the device limiter - this lets our reconstruction solver account for the wall's contribution to the total flux. Unfortunately, this can cause the cell to run for a long time of upwards of up to 10 minutes. Due to OpenFUSIONToolkit's incorporation of HODLR, running this cell is faster after running this notebook multiple times. 

In [ ]:
jam_obj = JAMfit(xml_name, meshfile_thincurr, meshfile_tokamaker, B0=1.91503608, R0=1.69550002, oft_env=myOFT)
jam_obj.setup_JAMfit("floops.loc", plot_files = save_path_dir)

## Doing a time-dependent synthetic run

Now that we have defined a JAMfit object, we can do a time-dependent synthetic run to find the necessary current eignmodes to represent the conducting structure's eddy currents within the reconstruction. In order to do that, we will need to feed in psuedo-data similar to the shot we want to reconstruct and perform a time-dependent synethetic run. In the following cell, we will use JAMfit's internal functions to create synthetic data for a time-dependent run. 

Note that the shot we are replicating is from "TokaMaker Example: Vertical stability in DIII-D from a gEQDSK equilibrium". 

In the following block we perform the sequential steps, 

1. Define the total plasma current over the selected time steps (`totalip`)
2. Define the change in time (`dt`) and the number of steps (`nsteps`) for the synthetic time run 
3. Define the points in time (`time_array`), note that it is advised for the time array to match `dt`*`nsteps` 
4. Define the coil currents over time, matching the ThinCurr and TokaMaker's coil definition
5. Use the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.setup_fil_timeseries "setup_fil_timeseries()" method to define a gaussian 2D plasma current profile over a poloidal slice over our R Z grid centered around a centroid defined by `r0` and `z0` with a gaussian spread defined by `sigma_r` and `sigma_z`. This obtains currents for every defined coil (shaping coils + plasma filaments) in the variable `final_coil_currs` 
6. Use the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.gen_synthetic_data "gen_synthetic_data()" method to perform the time dependent run. 

> **Warning:** While users can define their own coil currents as long as they have it within a numpy array with the shape (time points, coil currents) - where each time point is along the index 0 column, and the coil currents are in the same order as the xml file where each column corresponds to each coil with their current in each time point along each row - users must perform the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.gen_synthetic_data "gen_synthetic_data()" method to create a JAMfit reduced model. Which is necessary to do a JAMfit reconstruction.  


In [ ]:
# -- OVERALL SHOT SUMMARY -- 
totalip = [493324.5, 493324.5, 493324.5, 493324.5, 493324.5] # here we define the total plasma current over the shot
# in this example the ip doesnt change
dt = 2.E-4 # here we specify the change in time for the future time dependent run 
nsteps = 150 # here we specify the number of steps to take for our run in time 
time_array = np.array(np.linspace(0, 0.03, 5)).reshape(-1,1) # here we create the specific time points for our currents
# the time_array must be in (num_time_steps, 1) shape 
# note that it is advised for the time array to match the dt*steps 

# -- COIL CURRENTS -- 
# here we define the coil currents, matching the thincurr and tokamaker coil definitions 
f_coil_currents = {
'F1A' : 1999.5169719827586,
'F2A' : 1031.1705953663798,
'F3A' : -530.3450296336213,
'F4A' : -691.5127963362075,
'F5A' : 12.465672986260786,
'F6A' : -2142.877414772729,
'F7A' : 620.4805397727274,
'F8A' : -975.8820716594835,
'F9A' : 4223.563360873754,
'F1B' : 2213.251346982762,
'F2B' : 1316.7264278017242,
'F3B' : -385.27296605603476,
'F4B' : -2295.533943965518,
'F5B' : 6.891535265692348,
'F6B' : -2792.180113636364,
'F7B' : 754.6947443181815,
'F8B' : -886.0302397629307,
'F9B' : 4667.448286853513}

e_coil_currents = { 
'ECOILA' : -16030.96191406249,
'ECOILB' : -15782.150390625005,
'E567UP' : -16030.96191406249,
'E567DN' : -15782.150390625005,
'E89DN' : -15782.150390625005,
'E89UP' : -16030.96191406249}

coil_curr_temp = [] 
count = 0 
for i in f_coil_currents.keys(): 
    coil_curr_temp.append(f_coil_currents[i]) 
    count +=1 

for j in e_coil_currents.keys(): 
    coil_curr_temp.append(e_coil_currents[j])
coil_curr = np.tile(coil_curr_temp, (len(totalip), 1)) # now we have an array of coil currents 

# -- PLASMA CURRENT -- 
num_time_points = len(time_array)
r0_intial = 1.7
z0_intial = 0 
z0_final = -1.1 
r0_list = r0_intial*np.ones(num_time_points) 
z0_list = np.linspace(z0_intial, z0_final, num_time_points)
sigma_r = 2E-1
sigma_z = 2E-1 

# -- GENERATING SYNTHETIC DATA --  
# within the JAMfit class, the setup_fil_timeseries sets up a 2D gaussian profile given a r and z grid
# centered around a centroid defined by r0 and z0. You can simulate a VDE by moving r0 and z0 around over the num_time_steps
# sigma_r and sigma_z defines how peaked your gaussian profile would be (the lower the number the more peaked)

final_coil_currs = jam_obj.setup_fil_timeseries(time_array, totalip, coil_curr, r0_list, z0_list, sigma_r, sigma_z, R,Z)

# the gen_synthetic_data function then generates the time-dependent run 
# note that while the setup_fil_timeseries may not be necessary and users can define their own coil currents in the shape
# (time steps, currents) (where time steps are the column of index 0, and each column corresponds to a "coil" within JAMfit)
# the gen_synthetic_data function is necessary for creating a reduced model and continuing with the JAMfit reconstruction process 

jam_obj.gen_synthetic_data(final_coil_currs, dt, nsteps, verbose=True, s_freq = 1, p_freq=1 )

## Creating Reduced Object

Now that we have ran a time-dependent run, we can now get a reduced model based on the top 4 reduced eigenmodes and save it under the name 'DIIID_vde_reduced_model_ex.h5' using the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.create_from_runTD_top_modes "create_from_runTD_top_modes()". This method selects the top modes based on their weights over the time-dependent run. It also creates and saves a reduced model. Based on the time-dependent run, it outputs the sensor signals from the intialized sensors, coil currents (for each defined coil in the xml), the eigenvectors (when verbose=True), the top eigenmodes' indicies (when verbose=True), and their weights (when verbose=True).


**Note:** Users can create their own reduced models from manually selected modes using the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.create_reduced_model "create_reduced_model()" method. 

In [ ]:
wallmodes = 4
reduced_filename = 'DIIID_vde_reduced_model_ex.h5'

In [ ]:
reduced_obj, sensor_signals, currents, eig_vecs, eig_inds, weights = jam_obj.create_from_runTD_top_modes(wallmodes, reduced_filename, final_coil_currs, dt, nsteps, initial_num_eigs=35, verbose=True, s_freq = 1, p_freq=1)

## Saving Values for Reconstruction 

Here we are saving relevant values for the reconstruction which requires
1. total plasma current
2. coil currents
3. magnetic sensor measurements
4. time arrays (ideally they would all be in the same time-base but here we save each one as a precaution)

In [ ]:
num_coils = len(f_coil_currents.keys()) + len(e_coil_currents.keys())
time_array_coilcurr = currents['time']
coil_currs = currents['coil_curr'][:, 0:num_coils]
totalip_to_save = [] 
for i in range(currents['coil_curr'].shape[0]): 
    totalip_to_save.append(np.sum(currents['coil_curr'][i, num_coils:]))
time_array_sensors = sensor_signals['time']
num_real_sensors = 117
PsiFull = sensor_signals['sensors'][:, :num_real_sensors]

np.save('totalip.npy', totalip_to_save)
np.save('time_array_coilcurr.npy', time_array_coilcurr)
np.save('coil_currs.npy', coil_currs)
np.save('time_array_sensors.npy', time_array_sensors)
np.save('sensor_measurements.npy', PsiFull)

## Plotting for Visualization 
Here we plot for vizualization what the plasma current distribution looks like 

In [ ]:
plt.scatter(R,Z, c=currents['coil_curr'][0, num_coils:])
plt.plot(limiter[:,0],limiter[:,1],color='k')
plt.gca().set_aspect('equal', adjustable='box')